# calibrate_batch_dpwm notebook

This notebook embeds the full code from `calibrate_batch_dpwm.py` and runs `main()` directly.

In [ ]:
from pathlib import Path
import os

# Optional setup cell. Main code cell also sets project root.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)
print(f"Working directory: {ROOT}")

In [ ]:
from pathlib import Path
import os

# Notebook: ensure project root (run big cell alone).
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)

"""
Per-basin DPWM calibration for basins_selection catchments (resumable batch).

Uses:
  - Forcing: Results/forcing_per_basin/{basin_id}/rainplusmelt.csv, PET.csv
  - Parameter bounds: data/csv_estream/basins_selection.csv
  - Streamflow: data/streamflow/estreams_timeseries_streamflow_v02.csv (m3/s -> mm/day)
  - Common calibration period: 1976-01-01 to 2005-12-31

Default optimizer settings (same as six-basin workflow):
  invq_only, kge2012, inv_flow_offset=1.5, Sm in [0, 8000], warm_span=365,
  train_frac=0.70, max_gens=40, np_max=50

Pilot:
  python prepare_batch_forcing.py --limit 10 --skip_existing
  python calibrate_batch_dpwm.py --limit 10

Parallel (recommended on 10-core PC, leave 1-2 cores free):
  python calibrate_batch_dpwm.py --skip_existing --workers 8

Time-limited overnight run (9 h calibration after forcing is built):
  python prepare_batch_forcing.py --skip_existing --workers 8
  python calibrate_batch_dpwm.py --skip_existing --workers 8 --max_hours 9
"""

from __future__ import annotations

import argparse
import json
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd

from basin_selection_io import (
    DEFAULT_SELECTION_CSV,
    get_basin_row,
    list_basin_ids,
    load_basins_selection,
    parameter_bounds_from_selection,
)
from calibrate_validate_dpwm import lsade_optimize, objective_components
from dpwm_model import DPWM

DEFAULT_FORCING_DIR = Path("Results/forcing_per_basin")
DEFAULT_OUT_DIR = Path("Results/calibration_batch")
DEFAULT_STREAMFLOW = Path("data/streamflow/estreams_timeseries_streamflow_v02.csv")
DEFAULT_PERIOD_START = "1976-01-01"
DEFAULT_PERIOD_END = "2005-12-31"
MIN_VALID_DAYS = 50


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawTextHelpFormatter)
    p.add_argument("--selection_csv", type=Path, default=DEFAULT_SELECTION_CSV)
    p.add_argument("--forcing_dir", type=Path, default=DEFAULT_FORCING_DIR)
    p.add_argument("--streamflow_csv", type=Path, default=DEFAULT_STREAMFLOW)
    p.add_argument("--out_dir", type=Path, default=DEFAULT_OUT_DIR)
    p.add_argument("--period_start", default=DEFAULT_PERIOD_START)
    p.add_argument("--period_end", default=DEFAULT_PERIOD_END)
    p.add_argument("--limit", type=int, default=None)
    p.add_argument("--offset", type=int, default=0)
    p.add_argument("--basins", default="")
    p.add_argument("--skip_existing", action="store_true", help="Skip basins with params JSON already saved.")
    p.add_argument("--warm_span_days", type=int, default=365)
    p.add_argument("--train_frac", type=float, default=0.70)
    p.add_argument("--max_gens", type=int, default=40)
    p.add_argument("--np_max", type=int, default=50)
    p.add_argument("--np_min", type=int, default=10)
    p.add_argument("--memory_size", type=int, default=10)
    p.add_argument("--p_best_frac", type=float, default=0.2)
    p.add_argument("--max_warm_iterations", type=int, default=1000)
    p.add_argument("--inv_flow_offset", type=float, default=1.5)
    p.add_argument("--kge_mode", default="kge2012")
    p.add_argument("--objective_target", default="invq_only")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--sm_abs_min", type=float, default=0.0)
    p.add_argument("--sm_abs_max", type=float, default=8000.0)
    p.add_argument("--min_valid_days", type=int, default=MIN_VALID_DAYS)
    p.add_argument(
        "--workers",
        type=int,
        default=1,
        help="Parallel basin calibrations (default 1). Try 8 on a 10-core PC.",
    )
    p.add_argument(
        "--max_hours",
        type=float,
        default=None,
        help="Stop starting new basins after this many hours (in-flight jobs may finish).",
    )
    p.add_argument(
        "--no_require_forcing",
        action="store_true",
        help="Attempt calibration even if per-basin forcing CSVs are missing.",
    )
    return p.parse_known_args()[0]


def _args_to_config(args: argparse.Namespace) -> dict:
    return {
        "forcing_dir": str(args.forcing_dir),
        "streamflow_csv": str(args.streamflow_csv),
        "selection_csv": str(args.selection_csv),
        "out_dir": str(args.out_dir),
        "period_start": args.period_start,
        "period_end": args.period_end,
        "warm_span_days": args.warm_span_days,
        "train_frac": args.train_frac,
        "max_gens": args.max_gens,
        "np_max": args.np_max,
        "np_min": args.np_min,
        "memory_size": args.memory_size,
        "p_best_frac": args.p_best_frac,
        "max_warm_iterations": args.max_warm_iterations,
        "inv_flow_offset": args.inv_flow_offset,
        "kge_mode": args.kge_mode,
        "objective_target": args.objective_target,
        "seed": args.seed,
        "sm_abs_min": args.sm_abs_min,
        "sm_abs_max": args.sm_abs_max,
        "min_valid_days": args.min_valid_days,
    }


class _CalibArgs:
    """Lightweight namespace rebuilt inside worker processes."""

    def __init__(self, cfg: dict) -> None:
        for k, v in cfg.items():
            setattr(self, k, v)


def _calibrate_worker(task: dict) -> dict:
    """Run one basin; returns {status, basin_id, result|reason}."""
    basin_id = task["basin_id"]
    basin_index = task["basin_index"]
    cfg = task["config"]
    args = _CalibArgs(cfg)
    params_dir = Path(cfg["out_dir"]) / "params"
    param_path = params_dir / f"{basin_id}.json"

    try:
        precip, pet, dates = load_forcing(basin_id, Path(cfg["forcing_dir"]))
        period_start = pd.Timestamp(cfg["period_start"])
        period_end = pd.Timestamp(cfg["period_end"])
        mask = (dates >= period_start) & (dates <= period_end)
        if not np.all(mask):
            precip, pet, dates = precip[mask], pet[mask], dates[mask]
        if len(dates) == 0:
            raise ValueError("empty forcing after period filter")

        q_obs = load_streamflow_column(Path(cfg["streamflow_csv"]), basin_id, dates)
        selection_df = load_basins_selection(cfg["selection_csv"])
        row = get_basin_row(selection_df, basin_id)
        result = calibrate_one_basin(basin_id, row, precip, pet, q_obs, args, basin_index)
        param_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
        return {"status": "ok", "basin_id": basin_id, "result": result}
    except ValueError as exc:
        msg = str(exc)
        if "too_few_valid_days" in msg:
            return {"status": "excluded", "basin_id": basin_id, "reason": msg}
        return {"status": "failed", "basin_id": basin_id, "reason": msg}
    except Exception as exc:
        return {"status": "failed", "basin_id": basin_id, "reason": str(exc)}


def load_streamflow_column(streamflow_csv: Path, basin_id: str, dates: pd.DatetimeIndex) -> np.ndarray:
    """Load one basin column from the mega-CSV and align to dates (NaN if missing)."""
    chunks: list[pd.DataFrame] = []
    for chunk in pd.read_csv(streamflow_csv, usecols=["dates", basin_id], chunksize=100_000):
        chunk = chunk.rename(columns={"dates": "date", basin_id: "q_obs"})
        chunk["date"] = pd.to_datetime(chunk["date"]).dt.normalize()
        chunks.append(chunk)
    if not chunks:
        raise ValueError(f"Empty streamflow file: {streamflow_csv}")
    qdf = pd.concat(chunks, ignore_index=True)
    qdf = qdf.drop_duplicates(subset=["date"], keep="first")
    aligned = pd.DataFrame({"date": dates}).merge(qdf, on="date", how="left")
    return pd.to_numeric(aligned["q_obs"], errors="coerce").to_numpy(dtype=float)


def load_forcing(basin_id: str, forcing_dir: Path) -> tuple[np.ndarray, np.ndarray, pd.DatetimeIndex]:
    bdir = forcing_dir / basin_id
    rpm = pd.read_csv(bdir / "rainplusmelt.csv")
    pet = pd.read_csv(bdir / "PET.csv")
    rpm["date"] = pd.to_datetime(rpm["date"]).dt.normalize()
    pet["date"] = pd.to_datetime(pet["date"]).dt.normalize()
    merged = rpm.merge(pet, on="date", how="inner")
    dates = pd.DatetimeIndex(merged["date"])
    precip = merged["rainplusmelt"].to_numpy(dtype=float)
    pet_arr = merged["pet"].to_numpy(dtype=float)
    return precip, pet_arr, dates


def calibrate_one_basin(
    basin_id: str,
    selection_row: pd.Series,
    precip: np.ndarray,
    pet: np.ndarray,
    q_obs: np.ndarray,
    args: argparse.Namespace,
    basin_index: int,
) -> dict:
    bounds = parameter_bounds_from_selection(selection_row)
    area_km2 = bounds["area_km2"]
    q_obs_mm = q_obs * 86.4 / area_km2

    n = len(q_obs_mm)
    warm_span = args.warm_span_days
    valid_mask = np.isfinite(q_obs_mm) & np.isfinite(pet) & (np.arange(n) >= warm_span)
    valid_idx = np.where(valid_mask)[0]
    if valid_idx.size < args.min_valid_days:
        raise ValueError(
            f"too_few_valid_days: {valid_idx.size} < {args.min_valid_days} "
            f"(finite Q in window={int(np.sum(np.isfinite(q_obs_mm)))})"
        )

    rng = np.random.default_rng(args.seed + basin_index)
    n_train = int(round(args.train_frac * valid_idx.size))
    n_train = max(1, min(n_train, valid_idx.size - 1))
    train_idx = rng.choice(valid_idx, size=n_train, replace=False)
    val_idx = np.setdiff1d(valid_idx, train_idx, assume_unique=False)

    sm_low = max(1e-6, float(args.sm_abs_min))
    sm_high = max(sm_low + 1e-6, float(args.sm_abs_max))
    bounds_low = np.array(
        [bounds["alpha1_lo"], sm_low, bounds["alpha2_lo"], bounds["kf_lo"], bounds["ks_lo"]],
        dtype=float,
    )
    bounds_high = np.array(
        [bounds["alpha1_hi"], sm_high, bounds["alpha2_hi"], bounds["kf_hi"], bounds["ks_hi"]],
        dtype=float,
    )

    def vector_to_params(x: np.ndarray) -> list[float]:
        alpha1, sm, alpha2, kf, ks = map(float, x)
        return [alpha1, sm, alpha2, kf, ks]

    def eval_trial(x: np.ndarray) -> float:
        params = vector_to_params(x)
        model = DPWM(
            params,
            warm_span_days=warm_span,
            max_warm_iterations=args.max_warm_iterations,
        )
        with np.errstate(divide="ignore", invalid="ignore"):
            flux_output, _, _ = model.simulate(precip, pet)
        sim_q = flux_output.Q
        sim_et = flux_output.ET
        if not (np.all(np.isfinite(sim_q)) and np.all(np.isfinite(sim_et))):
            return float(1e6)
        comps = objective_components(
            sim_q, q_obs_mm, sim_et, pet, train_idx,
            inv_flow_offset=args.inv_flow_offset,
            kge_mode=args.kge_mode,
        )
        kge_invq = comps[1]
        return -float(kge_invq)

    t0 = time.perf_counter()
    best_x, best_f = lsade_optimize(
        eval_trial,
        bounds_low,
        bounds_high,
        max_gens=args.max_gens,
        np_max=args.np_max,
        np_min=args.np_min,
        memory_size=args.memory_size,
        p_best_frac=args.p_best_frac,
        seed=args.seed + 12345 * basin_index,
        progress_label=basin_id,
    )
    elapsed_s = time.perf_counter() - t0
    best_params = vector_to_params(best_x)

    return {
        "basin_id": basin_id,
        "status": "ok",
        "alpha1": best_params[0],
        "Sm": best_params[1],
        "alpha2": best_params[2],
        "Kf": best_params[3],
        "Ks": best_params[4],
        "KGE_invQ_cal": -float(best_f),
        "train_days": int(train_idx.size),
        "val_days": int(val_idx.size),
        "valid_days": int(valid_idx.size),
        "window_days": n,
        "area_km2": area_km2,
        "sm_prior_selection": bounds["sm_prior"],
        "objective_target": args.objective_target,
        "kge_mode": args.kge_mode,
        "inv_flow_offset": args.inv_flow_offset,
        "period_start": args.period_start,
        "period_end": args.period_end,
        "elapsed_s": round(elapsed_s, 1),
    }


def main() -> None:
    args = parse_args()
    args.out_dir.mkdir(parents=True, exist_ok=True)
    params_dir = args.out_dir / "params"
    params_dir.mkdir(parents=True, exist_ok=True)
    logs_dir = args.out_dir / "_batch_logs"
    logs_dir.mkdir(parents=True, exist_ok=True)

    period_start = pd.Timestamp(args.period_start)
    period_end = pd.Timestamp(args.period_end)

    selection_df = load_basins_selection(args.selection_csv)
    if args.basins.strip():
        basin_ids = [b.strip() for b in args.basins.split(",") if b.strip()]
    else:
        basin_ids = list_basin_ids(args.selection_csv, limit=args.limit, offset=args.offset)

    ok_rows: list[dict] = []
    excluded_rows: list[dict] = []
    failed_rows: list[dict] = []

    cfg = _args_to_config(args)
    pending: list[tuple[int, str]] = []
    skipped_no_forcing: list[str] = []
    for i, basin_id in enumerate(basin_ids, start=1):
        param_path = params_dir / f"{basin_id}.json"
        if args.skip_existing and param_path.exists():
            print(f"[{i}/{len(basin_ids)}] Skip {basin_id} (params exist)")
            continue
        bdir = args.forcing_dir / basin_id
        if not args.no_require_forcing and not (
            (bdir / "rainplusmelt.csv").exists() and (bdir / "PET.csv").exists()
        ):
            skipped_no_forcing.append(basin_id)
            continue
        pending.append((i, basin_id))

    if skipped_no_forcing:
        print(f"Skipped {len(skipped_no_forcing)} basins (forcing not built yet).")

    t_batch = time.perf_counter()
    workers = max(1, int(args.workers))
    max_seconds = float(args.max_hours) * 3600.0 if args.max_hours else None
    stopped_early = False

    def handle_outcome(out: dict) -> None:
        status = out["status"]
        bid = out["basin_id"]
        if status == "ok":
            result = out["result"]
            ok_rows.append(result)
            print(f"  OK {bid} KGE_invQ_cal={result['KGE_invQ_cal']:.4f} ({result['elapsed_s']}s)")
        elif status == "excluded":
            excluded_rows.append({"basin_id": bid, "reason": out["reason"]})
            print(f"  EXCLUDED {bid}: {out['reason']}")
        else:
            failed_rows.append({"basin_id": bid, "reason": out["reason"]})
            print(f"  FAILED {bid}: {out['reason']}")

    tasks = [
        {"basin_id": basin_id, "basin_index": i, "config": cfg}
        for i, basin_id in pending
    ]

    if workers == 1:
        for n_done, task in enumerate(tasks, start=1):
            if max_seconds is not None and (time.perf_counter() - t_batch) >= max_seconds:
                stopped_early = True
                print(f"Time limit ({args.max_hours} h) reached after {n_done - 1} basins.")
                break
            print(f"[{n_done}/{len(tasks)}] Calibrate {task['basin_id']} ...")
            handle_outcome(_calibrate_worker(task))
    else:
        limit_msg = f", max {args.max_hours} h" if max_seconds else ""
        print(f"Running up to {len(tasks)} basins with {workers} workers{limit_msg} ...")
        task_idx = 0
        done = 0
        with ProcessPoolExecutor(max_workers=workers) as pool:
            in_flight: dict = {}
            while task_idx < len(tasks) or in_flight:
                if max_seconds is not None and (time.perf_counter() - t_batch) >= max_seconds:
                    stopped_early = True
                    print(f"Time limit ({args.max_hours} h) reached. {len(tasks) - task_idx} not started.")
                    break
                while len(in_flight) < workers and task_idx < len(tasks):
                    if max_seconds is not None and (time.perf_counter() - t_batch) >= max_seconds:
                        stopped_early = True
                        break
                    t = tasks[task_idx]
                    task_idx += 1
                    fut = pool.submit(_calibrate_worker, t)
                    in_flight[fut] = t["basin_id"]
                if not in_flight:
                    break
                for fut in as_completed(list(in_flight.keys())):
                    bid = in_flight.pop(fut)
                    done += 1
                    print(f"[{done}] Finished {bid} ({task_idx}/{len(tasks)} started)")
                    handle_outcome(fut.result())
                    break

    if ok_rows:
        pd.DataFrame(ok_rows).to_csv(logs_dir / "calibration_ok.csv", index=False)
    if excluded_rows:
        pd.DataFrame(excluded_rows).to_csv(logs_dir / "calibration_excluded.csv", index=False)
    if failed_rows:
        pd.DataFrame(failed_rows).to_csv(logs_dir / "calibration_failed.csv", index=False)

    manifest = {
        "n_requested": len(basin_ids),
        "n_pending": len(pending),
        "n_skipped_no_forcing": len(skipped_no_forcing),
        "n_ok": len(ok_rows),
        "n_excluded": len(excluded_rows),
        "n_failed": len(failed_rows),
        "stopped_early_time_limit": stopped_early,
        "max_hours": args.max_hours,
        "workers": workers,
        "batch_elapsed_s": round(time.perf_counter() - t_batch, 1),
        "period": f"{args.period_start} .. {args.period_end}",
    }
    if skipped_no_forcing:
        pd.DataFrame({"basin_id": skipped_no_forcing}).to_csv(
            logs_dir / "calibration_skipped_no_forcing.csv", index=False
        )
    (logs_dir / "calibration_manifest.json").write_text(
        json.dumps(manifest, indent=2), encoding="utf-8"
    )
    print(
        f"\nBatch done in {manifest['batch_elapsed_s']}s. "
        f"OK={manifest['n_ok']} excluded={manifest['n_excluded']} failed={manifest['n_failed']}"
    )
    print(f"Logs: {logs_dir}")


# Execute the script entry point
main()
